# ¿Ensamblaje, red neuronal o LLM? Proyecto de Ecobici

Compara el mejor ensamblaje de la sesión pasada con una red neuronal simple, con la preparación y la división de [`pipeline_proyecto.py`](../../semana-09-sesion-02/plantilla-pipeline/pipeline_proyecto.py), y prepara la prueba del LLM sobre 10 filas de prueba.

Escribe `prompt_llm_proyecto.txt`. Los 10 números que respondió el LLM, uno por línea, están en `respuestas_llm_proyecto.txt`.

In [1]:
import os
import time

import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

import sys

# pipeline_proyecto.py vive en la carpeta del pipeline de la Semana 9, Sesion 2.
sys.path.insert(0, "../../semana-09-sesion-02/plantilla-pipeline")
from pipeline_proyecto import cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

# Las mismas 10 filas de prueba para las tres soluciones, y 30 filas de
# entrenamiento como ejemplos para el LLM.
muestra = X_prueba.sample(10, random_state=7)
y_muestra = y_prueba.loc[muestra.index]
ejemplos = X_ent.sample(30, random_state=7)
y_ejemplos = y_ent.loc[ejemplos.index]

Entrenamiento: (230400, 11) Prueba: (67100, 11)


## La función que evalúa cada modelo

In [2]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    mae_muestra = mean_absolute_error(y_muestra, pipeline.predict(muestra))
    resultados[nombre] = {
        "modelo": nombre,
        "mae_prueba": round(mae_prueba, 4),
        "mae_10_filas": round(mae_muestra, 4),
        "segundos": round(segundos, 3),
    }
    entrenados[nombre] = pipeline
    print(f"{nombre:20s} prueba {mae_prueba:7.4f}   10 filas {mae_muestra:7.4f}   {segundos:7.2f} s")

## El mejor ensamblaje de la sesión pasada

In [3]:
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))

gradient boosting    prueba  1.0278   10 filas  0.6670     18.60 s


## Una red neuronal simple

In [4]:
evaluar("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=200, early_stopping=True, random_state=42))

modelo_red = entrenados["red neuronal"].named_steps["modelo"]
resultados["red neuronal"]["parametros"] = sum(c.size for c in modelo_red.coefs_) + sum(i.size for i in modelo_red.intercepts_)
resultados["red neuronal"]["iteraciones"] = modelo_red.n_iter_

red neuronal         prueba  1.0209   10 filas  0.6691     40.05 s


## Tabla de los dos modelos

In [5]:
tabla = pd.DataFrame(resultados.values())
tabla.to_csv("comparacion_proyecto.csv", index=False)
tabla

,modelo,mae_prueba,mae_10_filas,segundos,parametros,iteraciones
0,gradient boosting,1.0278,0.6670,18.598,NaN,NaN
1,red neuronal,1.0209,0.6691,40.047,1025.0,45.0


## El prompt para el LLM

In [6]:
LEGIBLES = ["capacidad", "hora", "dia_semana", "es_fin_de_semana", "salidas",
            "llegadas", "salidas_lag_15m", "salidas_ultima_hora", "salidas_mismo_momento_ayer"]

filas_ejemplo = ejemplos[LEGIBLES].copy()
filas_ejemplo["salidas_prox_30m"] = y_ejemplos.values
texto = (
    "Estimas cuantas bicicletas saldran de una estacion de Ecobici en los proximos "
    "30 minutos. Cada fila es una estacion en una franja de 15 minutos: capacidad en "
    "anclajes, hora, dia_semana (1 es lunes, 7 es domingo), salidas y llegadas de la franja actual, "
    "salidas de la franja anterior, de la ultima hora y del mismo momento de ayer. "
    "Abajo hay 30 filas con su valor real y 10 sin el. Responde solo con 10 numeros, "
    "uno por linea, en el mismo orden, sin texto adicional. Si un valor falta, esta vacio.\n\n"
    "Filas con valor real:\n"
    + filas_ejemplo.to_csv(index=False)
    + "\nFilas sin valor:\n"
    + muestra[LEGIBLES].to_csv(index=False)
)
with open("prompt_llm_proyecto.txt", "w") as archivo:
    archivo.write(texto)
print("prompt_llm_proyecto.txt escrito:", len(texto), "caracteres")

prompt_llm_proyecto.txt escrito: 1944 caracteres


## Las respuestas del LLM

In [7]:
if os.path.exists("respuestas_llm_proyecto.txt"):
    with open("respuestas_llm_proyecto.txt") as archivo:
        predicciones_llm = [float(linea) for linea in archivo if linea.strip()]
    print("MAE del LLM en las 10 filas:", round(mean_absolute_error(y_muestra, predicciones_llm), 4))
else:
    print("Falta respuestas_llm_proyecto.txt")

MAE del LLM en las 10 filas: 0.4
